In [4]:
import cv2
import numpy as np
import os
from skimage.exposure import match_histograms


site_a_ref_path = 'angle1.png'
site_b_images = ['light1.png', 'light2.png', 'light3.png', 'angle2.png', 'angle3.png']

def preprocess_for_classifier(img_path, ref_img):
    # Check if file exists before reading
    if not os.path.exists(img_path):
        print(f"Skipping: {img_path} not found.")
        return None

    img = cv2.imread(img_path)

    # Resize to match reference dimensions
    img = cv2.resize(img, (ref_img.shape[1], ref_img.shape[0]))

    # LIGHTING STABILIZATION (The "Bridge" between Site A and Site B)
    # This forces Site B pixels to follow the brightness distribution of Site A
    img_stable_light = match_histograms(img, ref_img, channel_axis=-1)

    # FEATURE ENHANCEMENT
    gray = cv2.cvtColor(img_stable_light.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    return enhanced

# Execution Logic
reference_img = cv2.imread(site_a_ref_path)

if reference_img is None:
    print(f"Error: Could not load reference image {site_a_ref_path}. Check the filename.")
else:
    processed_dataset = []
    for img_path in site_b_images:
        processed = preprocess_for_classifier(img_path, reference_img)
        if processed is not None:
            processed_dataset.append(processed)
            # Save the result so you can see the "Stabilized" version
            cv2.imwrite(f'stabilized_{img_path}', processed)

    print(f"Successfully processed {len(processed_dataset)} images.")

Successfully processed 5 images.
